# La méthode de la puissance

| Module | Durée | Prérequis |
|---|---|---|
| M7 – Valeurs propres et dynamique | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une fédération de tennis de table veut un classement plus juste que le simple pourcentage de victoires : battre un joueur fort devrait rapporter davantage que battre un débutant. Elle adopte la **méthode de Keener** : le score d'un joueur est la moyenne de ses résultats contre chaque adversaire, **pondérés par le score de cet adversaire**.

Les scores dépendent donc les uns des autres. Si $B_{ij}$ résume les résultats du joueur $i$ contre le joueur $j$, le vecteur des scores $\mathbf{r}$ doit vérifier

$$
B\,\mathbf{r} = \lambda\,\mathbf{r},
$$

c'est-à-dire être un **vecteur propre** de $B$ : celui de la plus grande valeur propre, le seul dont tous les coefficients sont positifs.

Pour les 1 200 licenciés de la région, `np.linalg.eig` répond en une seconde environ. Mais la fédération nationale compte plus de 100 000 joueurs, et un nouveau classement doit être publié chaque semaine.

**Les questions : peut-on obtenir ce vecteur propre sans calculer tous les autres ? Combien de temps cela prend-il, et quand la méthode peut-elle échouer ?**

**Activité** : Au notebook 7.1, en appliquant plusieurs fois la même matrice, tous les vecteurs s'alignaient sur une direction. Comment cette observation pourrait-elle servir ici ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Le temps de calcul de `np.linalg.eig` est à peu près proportionnel à $n^3$ pour une matrice $n \times n$.

In [ ]:
pari_temps_eig_100000_joueurs = "1 minute"     # "1 minute", "1 heure", "1 jour" ou "1 an" (modifiez)
pari_iterations_necessaires = 1000             # pour la méthode de la puissance (modifiez)

## ③ Abstraire : la méthode de la puissance

On part d'un vecteur quelconque $\mathbf{x}_0$ et l'on applique $B$ encore et encore, en ramenant le résultat à la longueur 1 à chaque fois :

$$
\mathbf{x}_{k+1} = \frac{B\,\mathbf{x}_k}{\lVert B\,\mathbf{x}_k \rVert} .
$$

![Schéma : les vecteurs successifs x0, x1, x2, x4 tournent pour s'aligner sur la direction propre dominante ; le rythme est donné par le rapport des deux plus grandes valeurs propres](../../assets/schemas/schema-puissance.svg)

**Pourquoi ça marche.** Écrivons $\mathbf{x}_0 = c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \dots$ dans une base de vecteurs propres. Alors

$$
B^k\mathbf{x}_0 = c_1\lambda_1^k\mathbf{v}_1 + c_2\lambda_2^k\mathbf{v}_2 + \dots
= \lambda_1^k\left(c_1\mathbf{v}_1 + c_2\left(\frac{\lambda_2}{\lambda_1}\right)^k\mathbf{v}_2 + \dots\right).
$$

Si $\lvert \lambda_1 \rvert > \lvert \lambda_2 \rvert$, les termes parasites disparaissent comme $\lvert \lambda_2 / \lambda_1 \rvert^k$. Une fois $\mathbf{x}_k$ stabilisé, $\lVert B\mathbf{x}_k \rVert$ donne $\lvert \lambda_1 \rvert$.

**Le coût.** Chaque étape est un produit matrice-vecteur, soit environ $n^2$ opérations (bien moins si la matrice est creuse), au lieu de $n^3$ pour calculer tous les éléments propres.

**Activité** : Avec $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$ et $\mathbf{x}_0 = (1\,;\,0)$, calculez à la main $A\mathbf{x}_0$, $A^2\mathbf{x}_0$, $A^3\mathbf{x}_0$ (sans normaliser). Vers quelle direction vont-ils ? Que vaut le rapport des longueurs entre deux étapes ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### La ligue régionale

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def simuler_ligue(n, matchs_par_joueur=40, graine=0):
    # Simule une saison : chaque joueur a une force cachée ; le plus fort gagne plus souvent
    rng = np.random.default_rng(graine)
    force = rng.normal(0, 1, n)
    victoires, rencontres = np.zeros((n, n)), np.zeros((n, n))
    for _ in range(matchs_par_joueur * n // 2):
        i, j = rng.choice(n, 2, replace=False)
        if rng.random() < 1 / (1 + np.exp(force[j] - force[i])):
            victoires[i, j] += 1
        else:
            victoires[j, i] += 1
        rencontres[i, j] += 1
        rencontres[j, i] += 1
    return victoires, rencontres, force

def matrice_keener(victoires, rencontres):
    # Résultat ajusté contre chaque adversaire rencontré, moyenné sur le nombre d'adversaires
    B = np.where(rencontres > 0, (victoires + 1) / (rencontres + 2), 0.0)
    return B / (rencontres > 0).sum(axis=1, keepdims=True)

victoires, rencontres, force = simuler_ligue(1200)
B = matrice_keener(victoires, rencontres)
print("Taille de B :", B.shape, " coefficients non nuls :", f"{(B > 0).mean():.1%}")

### La méthode, programmée

In [ ]:
def puissance(B, tol=1e-10, max_iter=10_000):
    x = np.ones(B.shape[0]) / np.sqrt(B.shape[0])
    ecarts = []
    for k in range(1, max_iter + 1):
        y = B @ x
        lam = np.linalg.norm(y)
        y = y / lam
        ecarts.append(np.linalg.norm(y - x))
        if ecarts[-1] < tol:
            return lam, y, k, ecarts
        x = y
    raise RuntimeError(f"Pas de convergence en {max_iter} itérations")

debut = time.perf_counter()
lam, r, iterations, ecarts = puissance(B)
t_puissance = time.perf_counter() - debut

debut = time.perf_counter()
valeurs, vecteurs = np.linalg.eig(B)
t_eig = time.perf_counter() - debut
k = np.argmax(np.abs(valeurs))
v = np.abs(vecteurs[:, k].real)

print(f"Méthode de la puissance : lambda = {lam:.6f} en {iterations} itérations, {1000 * t_puissance:.1f} ms")
print(f"np.linalg.eig           : lambda = {abs(valeurs[k]):.6f}, {1000 * t_eig:.0f} ms")
print(f"Écart maximal entre les deux vecteurs normés : {np.abs(r - v / np.linalg.norm(v)).max():.1e}")

### Le classement obtenu

In [ ]:
pourcentage = victoires.sum(axis=1) / rencontres.sum(axis=1)
rang_keener = np.argsort(-r)
print("Les 8 premiers selon Keener :")
for place, i in enumerate(rang_keener[:8], 1):
    print(f"  {place}. joueur {i:4d}   victoires {pourcentage[i]:.0%}   force cachée {force[i]:+.2f}")
print(f"\nCorrélation avec la force cachée : Keener {np.corrcoef(r, force)[0, 1]:.3f}, "
      f"pourcentage de victoires {np.corrcoef(pourcentage, force)[0, 1]:.3f}")

### Le temps de calcul selon la taille

In [ ]:
tailles = [200, 400, 800, 1200]
temps_eig, temps_puissance = [], []
for n in tailles:
    Bn = matrice_keener(*simuler_ligue(n, graine=n)[:2])
    debut = time.perf_counter(); np.linalg.eig(Bn); temps_eig.append(time.perf_counter() - debut)
    debut = time.perf_counter(); puissance(Bn); temps_puissance.append(time.perf_counter() - debut)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.loglog(tailles, temps_eig, "o-", color=LIOUAI["navy"], label="np.linalg.eig")
ax.loglog(tailles, temps_puissance, "s-", color=LIOUAI["vert"], label="méthode de la puissance")
ax.set_xlabel("nombre de joueurs n"); ax.set_ylabel("temps (s)"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

pente = np.polyfit(np.log(tailles), np.log(temps_eig), 1)[0]
extrapolation = temps_eig[-1] * (100_000 / tailles[-1]) ** 3
print(f"Pente mesurée pour eig : {pente:.1f} ; extrapolation à 100 000 joueurs en n³ : {extrapolation / 3600:.0f} heures")

**Activité** : Pourquoi la pente mesurée n'est-elle pas exactement 3 ? Combien de mémoire faudrait-il pour stocker une matrice $100\,000 \times 100\,000$ de nombres à virgule (8 octets chacun) ? Cette contrainte change-t-elle la conclusion ?

Indiquez vos réponses et justifiez-les.

### La vitesse de convergence dépend de $\lambda_2 / \lambda_1$

Comparons avec une fédération formée de deux ligues qui ne se rencontrent presque jamais.

In [ ]:
def deux_ligues(n=600, croisements=20, graine=3):
    v, m, _ = simuler_ligue(n, graine=graine)
    moitie = n // 2
    masque = np.zeros((n, n), dtype=bool)
    masque[:moitie, :moitie] = masque[moitie:, moitie:] = True
    rng = np.random.default_rng(graine)
    for _ in range(croisements):                  # quelques rencontres entre les deux ligues
        i, j = rng.integers(0, moitie), rng.integers(moitie, n)
        masque[i, j] = masque[j, i] = True
    return v * masque, m * masque

B2 = matrice_keener(*deux_ligues())
fig, ax = plt.subplots(figsize=(8, 3.8))
for M, nom, couleur in [(B, "une ligue", LIOUAI["vert"]), (B2, "deux ligues séparées", LIOUAI["navy"])]:
    modules = np.sort(np.abs(np.linalg.eigvals(M)))[::-1]
    _, _, it, ec = puissance(M, max_iter=50_000)
    ax.loglog(range(1, len(ec) + 1), ec, color=couleur, label=f"{nom} : |λ2/λ1| = {modules[1] / modules[0]:.3f}, {it} itérations")
ax.set_xlabel("itération"); ax.set_ylabel("écart entre deux itérés"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Déboguer : trois façons d'échouer

Les trois cellules suivantes contiennent des versions de la méthode qui échouent. Pour chacune, observez le symptôme et cherchez la cause.

In [ ]:
# 1. On oublie de normaliser
x = np.ones(len(B))
grand = 50 * B                         # même matrice, valeurs propres multipliées par 50
for k in range(300):
    x = grand @ x
print("Avec λ1 ≈ 25, sans normalisation, après 300 étapes :", x[:3])
x = np.ones(len(B))
for k in range(1500):
    x = B @ x
print("Avec λ1 ≈ 0,49, sans normalisation, après 1500 étapes :", x[:3])

In [ ]:
# 2. Une valeur propre dominante négative
C = np.array([[-3., 1], [1, 2]])
print("Valeurs propres :", np.real_if_close(np.linalg.eigvals(C)).round(3))
x = np.array([1., 0])
for k in range(6):
    x = C @ x
    x = x / np.linalg.norm(x)
    print(f"étape {k + 1} : x = {x.round(4)}")

In [ ]:
# 3. Deux valeurs propres de même module
D = np.array([[0., 1], [1, 0]])
print("Valeurs propres :", np.real_if_close(np.linalg.eigvals(D)).round(3))
x = np.array([0.9, 0.1])
for k in range(4):
    x = D @ x
    print(f"étape {k + 1} : x = {x}")

**Activité** : Pour chaque échec, expliquez la cause avec les valeurs propres. Proposez une correction pour le premier et le deuxième. Pourquoi le troisième ne peut-il pas être corrigé simplement ? Pourquoi les matrices de Keener, à coefficients positifs, ne présentent-elles pas ces problèmes ?

Indiquez vos réponses et justifiez-les.

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : eig sur 100 000 joueurs en {pari_temps_eig_100000_joueurs} ; {pari_iterations_necessaires} itérations")
print(f"Mesures   : environ {extrapolation / 3600:.0f} heures (et une mémoire impossible) ; {iterations} itérations ici")

**Activité** :

1. Rédigez pour la fédération une recommandation : quel outil utiliser pour le classement national, et pourquoi ?
2. Le classement de Keener est-il meilleur que le pourcentage de victoires ? Sur quoi vous fondez-vous ? Quelle information ce classement utilise-t-il en plus ?
3. Dans quel cas la méthode de la puissance deviendrait-elle lente ? Que faudrait-il organiser dans le calendrier des rencontres pour l'éviter ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Tous les éléments propres ?** La méthode ne donne que le vecteur propre dominant. Quand a-t-on besoin des autres ? (Le module M8 en donnera un exemple.)
- **Le critère d'arrêt.** On s'arrête quand deux itérés sont très proches. Ce critère garantit-il que le vecteur est précis ? Que se passe-t-il quand $\lvert \lambda_2/\lambda_1 \rvert$ est très proche de 1 ?
- **Généraliser : ne jamais construire la matrice.** Pour 100 000 joueurs, la matrice $B$ ne tient pas en mémoire, mais chaque joueur n'a que 40 adversaires. Écrivez une fonction `produit_keener(i, j, poids, x)` qui calcule $B\mathbf{x}$ à partir de la seule **liste** des paires de joueurs qui se sont rencontrés (`i`, `j`) et de leurs coefficients `poids`, sans créer de matrice $n \times n$. Vérifiez qu'elle donne le même résultat que `B @ x`, puis utilisez-la dans une méthode de la puissance.

In [ ]:
def produit_keener(i, j, poids, x):
    # À compléter : renvoie le vecteur B x, où B[i[k], j[k]] = poids[k]
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def produit_keener(i, j, poids, x):
    resultat = np.zeros_like(x)
    np.add.at(resultat, i, poids * x[j])      # ajoute poids * x[j] à la case i, pour chaque paire
    return resultat

i, j = np.nonzero(B)
poids = B[i, j]
x = np.random.default_rng(0).random(len(B))
print("Même résultat que B @ x ?", np.allclose(produit_keener(i, j, poids, x), B @ x))
print(f"Mémoire : matrice pleine {B.nbytes / 1e6:.1f} Mo, liste {(i.nbytes + j.nbytes + poids.nbytes) / 1e6:.1f} Mo")

def puissance_liste(i, j, poids, n, tol=1e-10):
    x = np.ones(n) / np.sqrt(n)
    for k in range(1, 10_000):
        y = produit_keener(i, j, poids, x)
        lam = np.linalg.norm(y)
        y /= lam
        if np.linalg.norm(y - x) < tol:
            return lam, y, k
        x = y

lam_l, r_l, k_l = puissance_liste(i, j, poids, len(B))
print(f"Sans matrice : lambda = {lam_l:.6f} en {k_l} itérations ; même classement ? {np.allclose(r_l, r)}")
print(f"Pour 100 000 joueurs à 40 adversaires : environ {100_000 * 40 * 20 / 1e6:.0f} Mo, au lieu de {100_000 ** 2 * 8 / 1e9:.0f} Go")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Optimiser la rapidité et la réutilisabilité du calcul": 1,
    "Résoudre un problème de fonctionnement": 1,
    "Comparer les mérites des outils dans le contexte": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Corroborer ses résultats": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "7.4-methode-de-la-puissance", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")